# Exercise: Feature Engineering for Trading Models 

In this exercise, you'll get practice engineering features for trading models. You can use built-in Pandas methods to do this feature engineering. In the demo, we'll show you how to use a more specialized library called `ta` to do technical analysis feature engineering. 

In [ ]:
import pandas as pd
import yfinance as yf

**Pull data for one stock ticker from YahooFinance**

Use the YF API to pull daily price data for at least 2 years for any stock ticker you'd like.

In [ ]:
stock_ticker = "000660.KS"
start_date = "2025-01-01"  # use format YYYY-MM-DD
end_date = "2026-05-10"

In [ ]:
data = yf.download(
    tickers=stock_ticker, start=start_date, end=end_date
)  # replace ... inside this function with the correct parameters in order to get your data

In [ ]:
data.head()

**Calculate the 10-day momentum for the above ticker's closing price**

Recall that the 10-day momentum is the rate of change of a price over a 10-day period. It's used in technical analysis to see in which direction and with what magnitude an asset's price is moving. 

To calculate the rate of change, recall you can use the Pandas method `pct_change()`. To get a 10-day rate of change speficially, you'll have to pass in some parameter to the `pct_change()` method. Reading the documentation for that method may help: 



In [ ]:
data["10_day_momentum"] = data['Close'].rolling(window=10).mean()

In [ ]:
data.head(20)

**Calculate a 12-day and 26-day exponential moving average**

Using the closing price for your stock, use Pandas to calculate a 12-day and 26-day EMA (exponential moving average). Look into the Pandas method `ewm()`, which was used in the demo. 

In [ ]:
data["EMA_12"] = data['Close'].ewm(span=12).mean()
data["EMA_26"] = data['Close'].ewm(span=26).mean()

In [ ]:
data.head(20)

**Manually calculate the MACD (moving average convergence divergence)**

Recall that the MACD is calculated as the 12-day exponential moving average minus the 26-day. Use the above step to calculate the MACD. You'll have to create your own column for this step. 


In [ ]:
# calculate the MACD and save it to a new column in your dataframe
data['MACD'] = data['EMA_12'] - data['EMA_26']

In [ ]:
data.head(20)

**Manually calculate the MACD Signal**

Recall that the MACD signal (discussed in the feature engineering demo) is calculated as the 9-period exponential moving average of the MACD (calculated in the prior step). Can you manually use Pandas methods to calculate the MACD signal? Create a new column for it in your dataframe. 

In [ ]:
# calculate the MACD signal using the above MACD using only Pandas methods (don't use the the library shown in the demo)
data['MACD signal'] = data['MACD'].ewm(span=9).mean()


In [ ]:
data = data.dropna()

In [ ]:
data.head(20)

**Visualize Closing Price, EMAs, MACD and Signal Line**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

# --- Top plot: Closing price + EMAs ---
ax1.plot(data.index, data['Close'], label='Close', color='black', linewidth=1.2)
ax1.plot(data.index, data['EMA_12'], label='EMA 12', color='blue', linewidth=1, linestyle='--')
ax1.plot(data.index, data['EMA_26'], label='EMA 26', color='orange', linewidth=1, linestyle='--')
ax1.set_title(f'{stock_ticker} — Closing Price & EMAs')
ax1.set_ylabel('Price')
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Bottom plot: MACD + Signal ---
ax2.plot(data.index, data['MACD'], label='MACD', color='blue', linewidth=1.2)
ax2.plot(data.index, data['MACD signal'], label='Signal (9-day EMA)', color='red', linewidth=1, linestyle='--')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='-')

# Histogram bars coloured by sign
colors = ['green' if v >= 0 else 'red' for v in (data['MACD'] - data['MACD signal'])]
ax2.bar(data.index, data['MACD'] - data['MACD signal'], color=colors, alpha=0.4, label='Histogram')

ax2.set_title('MACD & Signal Line')
ax2.set_ylabel('MACD')
ax2.set_xlabel('Date')
ax2.legend()
ax2.grid(True, alpha=0.3)

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
